# Review Entity Sentiment Loader

Select `METHOD = "rule"` or `METHOD = "model"` to compute review-level sentiment and write it to `review_entity_sentiment`. Sentiment is computed per review and duplicated across each linked `review_entity` row. The table stores raw outputs only.

In [20]:
import sqlite3
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd

DB_PATH = Path("../data/app.sqlite")
MODEL_NAME = "cardiffnlp/twitter-xlm-roberta-base-sentiment"

In [21]:
RULE_VERSION = "v1"

#METHOD = "rule"
METHOD = "model"

In [22]:
import re

LEXICON = {
    "good": 1.0,
    "great": 1.5,
    "amazing": 2.0,
    "excellent": 2.0,
    "love": 1.5,
    "like": 1.0,
    "bad": -1.0,
    "terrible": -2.0,
    "awful": -2.0,
    "hate": -1.5,
    "dislike": -1.0,
    "poor": -1.0,
    "worst": -2.0,
    "좋다": 1.0,
    "최고": 2.0,
    "훌륭": 1.5,
    "맛있": 1.5,
    "추천": 1.0,
    "만족": 1.0,
    "친절": 1.0,
    "별로": -1.0,
    "나쁘": -1.0,
    "최악": -2.0,
    "불친절": -1.5,
    "실망": -1.0,
}


def _tokenize(text: str) -> list[str]:
    return re.findall(r"[A-Za-z']+|[가-힣]+", (text or "").lower())


def compute_rule_sentiment(text: str) -> tuple[float, float, str, str]:
    tokens = _tokenize(text)
    if not tokens:
        return 0.0, 0.0, "rule", RULE_VERSION
    score = 0.0
    matched = 0
    for token in tokens:
        weight = LEXICON.get(token)
        if weight is None:
            continue
        matched += 1
        score += weight
    confidence = matched / len(tokens)
    return score, confidence, "rule", RULE_VERSION


def load_model_pipeline():
    from transformers import AutoModelForSequenceClassification, AutoTokenizer, pipeline

    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME,
        use_safetensors=True,
    )
    return pipeline(
        "sentiment-analysis",
        model=model,
        tokenizer=tokenizer,
        truncation=True,
        max_length=256,
        batch_size=16,
    )


def compute_model_sentiment(text: str, pipe) -> tuple[float, float, str, str]:
    result = pipe(text, truncation=True)[0]
    label = str(result.get("label", "")).lower()
    score = float(result.get("score", 0.0))
    if "negative" in label:
        polarity = -score
    elif "positive" in label:
        polarity = score
    else:
        polarity = 0.0
    return polarity, score, "model", MODEL_NAME

In [23]:
if not DB_PATH.exists():
    raise FileNotFoundError(f"DB not found at {DB_PATH.resolve()}")

conn = sqlite3.connect(DB_PATH)
conn.execute("PRAGMA foreign_keys = ON")

def ensure_review_entity_sentiment(conn):
    has_table = conn.execute(
        "SELECT name FROM sqlite_master WHERE type='table' AND name='review_entity_sentiment'"
    ).fetchone()
    schema = """
    CREATE TABLE IF NOT EXISTS review_entity_sentiment (
      review_entity_id INTEGER NOT NULL,
      sentiment_raw REAL NOT NULL,
      confidence REAL NOT NULL,
      method TEXT NOT NULL,
      version TEXT,
      created_at TEXT,
      PRIMARY KEY (review_entity_id, method, version),
      FOREIGN KEY (review_entity_id) REFERENCES review_entity(id)
    );
    """
    if not has_table:
        conn.executescript(schema)
        return
    fk_rows = conn.execute("PRAGMA foreign_key_list('review_entity_sentiment')").fetchall()
    if any(row[2] == "review_entity_old" for row in fk_rows):
        conn.execute("PRAGMA foreign_keys = OFF")
        conn.executescript(
            """
            ALTER TABLE review_entity_sentiment RENAME TO review_entity_sentiment_old;
            CREATE TABLE review_entity_sentiment (
              review_entity_id INTEGER NOT NULL,
              sentiment_raw REAL NOT NULL,
              confidence REAL NOT NULL,
              method TEXT NOT NULL,
              version TEXT,
              created_at TEXT,
              PRIMARY KEY (review_entity_id, method, version),
              FOREIGN KEY (review_entity_id) REFERENCES review_entity(id)
            );
            INSERT OR REPLACE INTO review_entity_sentiment
            (review_entity_id, sentiment_raw, confidence, method, version, created_at)
            SELECT review_entity_id, sentiment_raw, confidence, method, version, created_at
            FROM review_entity_sentiment_old;
            DROP TABLE review_entity_sentiment_old;
            """
        )
        conn.execute("PRAGMA foreign_keys = ON")


ensure_review_entity_sentiment(conn)

review_entity_columns = pd.read_sql("PRAGMA table_info('review_entity')", conn)
review_entity_id_col = "id" if "id" in review_entity_columns["name"].tolist() else "rowid"
review_entity = pd.read_sql(
    f"SELECT {review_entity_id_col} AS review_entity_id, review_id FROM review_entity",
    conn,
)
reviews = pd.read_sql(
    "SELECT id AS review_id, content FROM review",
    conn,
)

if review_entity.empty:
    raise ValueError("review_entity is empty; nothing to compute.")

if METHOD == "model":
    pipe = load_model_pipeline()
    sentiment = reviews["content"].apply(lambda text: compute_model_sentiment(text, pipe))
elif METHOD == "rule":
    sentiment = reviews["content"].apply(compute_rule_sentiment)
else:
    raise ValueError(f"Unknown METHOD: {METHOD}")

reviews["sentiment_raw"] = sentiment.map(lambda item: item[0])
reviews["confidence"] = sentiment.map(lambda item: item[1])
reviews["method"] = sentiment.map(lambda item: item[2])
reviews["version"] = sentiment.map(lambda item: item[3])

rows = review_entity.merge(
    reviews[["review_id", "sentiment_raw", "confidence", "method", "version"]],
    on="review_id",
    how="left",
)
rows["created_at"] = datetime.now(timezone.utc).isoformat()

payload = list(
    rows[
        [
            "review_entity_id",
            "sentiment_raw",
            "confidence",
            "method",
            "version",
            "created_at",
        ]
    ].itertuples(index=False, name=None)
)

conn.executemany(
    """
    INSERT OR REPLACE INTO review_entity_sentiment
    (review_entity_id, sentiment_raw, confidence, method, version, created_at)
    VALUES (?, ?, ?, ?, ?, ?)
    """,
    payload,
)
conn.commit()
conn.close()

print(f"Inserted/updated {len(payload)} sentiment rows using METHOD={METHOD}.")

Device set to use cpu


Inserted/updated 49 sentiment rows using METHOD=model.
